# Phase 6A — PyTorch Foundations

**Theory:** Tensors, autograd, neural network layers, loss functions, gradient descent, training loops.

**Install:** `pip install torch torchvision`
- CPU-only: `pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu`
- With CUDA: see https://pytorch.org/get-started/locally/

---

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset

    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available : {torch.cuda.is_available()}")
    TORCH_AVAILABLE = True
except ImportError:
    print("PyTorch not installed. Run: pip install torch torchvision")
    TORCH_AVAILABLE = False

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

---
## 1. Tensors — The Fundamental Data Structure

In [ ]:
if TORCH_AVAILABLE:
    # Tensors are like numpy arrays but can run on GPU and support autograd

    # Creating tensors
    t1 = torch.tensor([1.0, 2.0, 3.0])  # from list
    t2 = torch.zeros(3, 4)  # 3x4 of zeros
    t3 = torch.randn(2, 3)  # random normal
    t4 = torch.arange(0, 10, dtype=torch.float32)  # like np.arange

    print(f"t1: {t1},  dtype={t1.dtype},  shape={t1.shape}")
    print(f"t3:\n{t3}")

    # NumPy ↔ PyTorch conversion
    np_arr = np.array([1.0, 2.0, 3.0])
    tensor_from_np = torch.from_numpy(np_arr)
    back_to_np = tensor_from_np.numpy()
    print(f"\nNumPy array:    {np_arr}")
    print(f"PyTorch tensor: {tensor_from_np}")
    print(f"Back to NumPy:  {back_to_np}")

In [ ]:
if TORCH_AVAILABLE:
    # Tensor operations — identical to NumPy
    a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
    b = torch.tensor([[5.0, 6.0], [7.0, 8.0]])

    print("Addition:\n", a + b)
    print("Matrix multiply:\n", torch.mm(a, b))
    print("Mean:", a.mean())
    print("Shape:", a.shape)
    print("Reshape:", a.reshape(1, 4))
    print("Transpose:\n", a.T)

---
## 2. Autograd — Automatic Differentiation

PyTorch tracks operations on tensors and computes gradients automatically. This is how backpropagation works.

In [ ]:
if TORCH_AVAILABLE:
    # requires_grad=True tells PyTorch to track this tensor
    x = torch.tensor([2.0, 3.0], requires_grad=True)

    # Forward pass: compute y = sum(x^2 + 3x + 1)
    y = (x**2 + 3 * x + 1).sum()
    print(f"y = {y.item():.2f}")

    # Backward pass: compute dy/dx
    y.backward()

    # Gradient: dy/dx = 2x + 3
    print(f"Gradients: {x.grad}")  # [2*2+3, 2*3+3] = [7, 9]

---
## 3. Building a Neural Network with nn.Module

In [ ]:
if TORCH_AVAILABLE:

    class MLP(nn.Module):
        """Multi-Layer Perceptron for binary classification."""

        def __init__(self, input_dim, hidden_dim, output_dim):
            super().__init__()
            self.network = nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(p=0.3),
                nn.Linear(hidden_dim, hidden_dim // 2),
                nn.ReLU(),
                nn.Linear(hidden_dim // 2, output_dim),
            )

        def forward(self, x):
            return self.network(x)

    model = MLP(input_dim=10, hidden_dim=64, output_dim=1)
    print(model)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\nTotal parameters: {total_params}")

---
## 4. The Training Loop

The core PyTorch pattern:
```
for epoch in range(n_epochs):
    for batch in dataloader:
        optimizer.zero_grad()  # clear old gradients
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()        # compute gradients
        optimizer.step()       # update weights
```

In [ ]:
if TORCH_AVAILABLE:
    from sklearn.datasets import make_classification
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler

    # Data
    X_np, y_np = make_classification(n_samples=1000, n_features=10, random_state=42)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_np, y_np, test_size=0.2, random_state=42
    )

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_te = scaler.transform(X_te)

    # Convert to tensors
    X_train_t = torch.FloatTensor(X_tr)
    y_train_t = torch.FloatTensor(y_tr).unsqueeze(1)
    X_test_t = torch.FloatTensor(X_te)
    y_test_t = torch.FloatTensor(y_te).unsqueeze(1)

    # DataLoader (batching)
    train_ds = TensorDataset(X_train_t, y_train_t)
    train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)

    # Model, loss, optimizer
    model = MLP(input_dim=10, hidden_dim=64, output_dim=1)
    criterion = nn.BCEWithLogitsLoss()  # binary cross-entropy
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Training loop
    train_losses = []
    n_epochs = 50

    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0.0
        for X_batch, y_batch in train_dl:
            optimizer.zero_grad()
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        train_losses.append(epoch_loss / len(train_dl))

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch + 1:3d}/{n_epochs}  Loss: {train_losses[-1]:.4f}")

    # Evaluation
    model.eval()
    with torch.no_grad():
        logits = model(X_test_t)
        preds = (torch.sigmoid(logits) > 0.5).float()
        acc = (preds == y_test_t).float().mean().item()
    print(f"\nTest accuracy: {acc:.4f}")

    # Plot training loss
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(train_losses, color="steelblue")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Training Loss Curve")
    plt.tight_layout()
    plt.show()

---
## 5. Activation Functions

In [ ]:
# Visualize activation functions (no PyTorch needed for this plot)
x = np.linspace(-4, 4, 200)

activations = {
    "ReLU": lambda x: np.maximum(0, x),
    "Sigmoid": lambda x: 1 / (1 + np.exp(-x)),
    "Tanh": lambda x: np.tanh(x),
    "Leaky ReLU": lambda x: np.where(x > 0, x, 0.1 * x),
}

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, (name, fn) in zip(axes.flat, activations.items()):
    ax.plot(x, fn(x), "steelblue", lw=2)
    ax.axhline(0, color="gray", lw=0.8, linestyle=":")
    ax.axvline(0, color="gray", lw=0.8, linestyle=":")
    ax.set_title(name)
    ax.set_ylim(-1.5, 4)

plt.suptitle("Activation Functions", fontsize=13)
plt.tight_layout()
plt.show()

print("\nWhen to use:")
print("  ReLU      : Hidden layers — fast, default choice")
print("  Sigmoid   : Output layer for binary classification")
print("  Softmax   : Output layer for multi-class classification")
print("  Tanh      : Recurrent networks (LSTM, GRU)")
print("  Leaky ReLU: When neurons die (output always 0) with ReLU")

---
## Summary

| Concept | PyTorch | Note |
|---------|---------|------|
| Tensor creation | `torch.tensor()`, `torch.randn()`, `torch.zeros()` | Like numpy array |
| Enable gradient | `requires_grad=True` | For parameters |
| Clear gradients | `optimizer.zero_grad()` | EVERY batch, before backward |
| Forward pass | `model(x)` | Calls `forward()` |
| Compute loss | `criterion(pred, target)` | BCEWithLogits, CrossEntropy, MSE |
| Backward | `loss.backward()` | Fills `.grad` on all parameters |
| Update weights | `optimizer.step()` | Adam, SGD, RMSProp |
| Eval mode | `model.eval()` + `torch.no_grad()` | Disables dropout, gradient tracking |